# Crisis Analysis, Briefing Generation & Safety (Person 3)

Turns Person 2's grounded briefings into:
1. a comparison with the previous briefing (what changed, trends)
2. a situation report grouped by certainty (confirmed / preliminary / conflicting / not available)
3. an executive briefing (Markdown)
4. a public statement draft that passes safety checks
5. a human approval step before anything can be released

This step makes no LLM calls. See `docs/briefing-analysis-and-safety.md` for the full design.

## Settings
`USE_LLM = False` runs without an API key: briefings use the source text instead of model summaries. Set it to `True` to use `ANTHROPIC_API_KEY` from `.env`.

In [1]:
import contextlib, json, sys
from pathlib import Path
from unittest import mock
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.crisis import briefing as b
from src.crisis.reporting import (build_situation_report, render_executive_briefing, draft_public_statement,
                                  check_release_safety, edit_draft, approve, reject, release, ReleaseBlocked)

USE_LLM = False
IGNORE_AGE = True   # the saved snapshots are from Nov 2026; True skips the "too old" check so they can be used
MAX_EVENTS = None

## 1. Build the previous and current briefings (Person 2's step)

In [2]:
files = sorted((ROOT / "data" / "live").glob("validated-output-*.json"))
prev_batch, cur_batch = (json.loads(f.read_text()) for f in (files[0], files[-1]))
print("previous:", files[0].name, "\ncurrent: ", files[-1].name)

llm = b.get_llm() if USE_LLM else (lambda s, u, sc: (_ for _ in ()).throw(RuntimeError("LLM disabled")))
with mock.patch.object(b, "_still_current", lambda r: True) if IGNORE_AGE else contextlib.nullcontext():
    previous = b.build_briefing(prev_batch, llm, MAX_EVENTS)
    current = b.build_briefing(cur_batch, llm, MAX_EVENTS)
print(len(previous["events"]), "->", len(current["events"]), "events")

previous: validated-output-live_20261107_075036Z.json 
current:  validated-output-live_20261111_201200Z.json
17 -> 15 events


## 2. Situation report and changes since the last briefing

In [3]:
report = build_situation_report(current, previous, batch=cur_batch, previous_batch=prev_batch)
print(json.dumps(report["counts"], indent=2))
print("\n".join(report["change_summary"]))

{
  "events": 15,
  "priority": 3,
  "confirmed": 0,
  "preliminary": 15,
  "conflicting": 0
}
14 new events, including: weather alert | Montgomery; weather alert | South-Central Shasta County Including Redding Metro, Northern Sacramento Valley in Tehama County, Central Glenn County Including Interstate 5 and 2 more; weather alert | Surprise Valley California, Northern Washoe County; weather alert | Burke, Renville, Northwest Ward and 1 more; weather alert | Grand Traverse Bay south of a line Grand Traverse Light to Norwood MI, Norwood MI to 5NM West of Mackinac Bridge including Little Traverse Bay, Sleeping Bear Point to Grand Traverse Light MI and 2 more
16 ended events, including: weather alert | Montgomery; weather alert | Clatsop County Coast, Tillamook County Coast, South Washington Coast; weather alert | Mercer, Putnam, Schuyler and 13 more; weather alert | Atchison, Miami, Linn and 23 more; weather alert | Sodus Bay to Mexico Bay NY, Mexico Bay NY to the St. Lawrence River
decl

## 3. Executive briefing

In [4]:
display(Markdown(render_executive_briefing(report)))

# Executive Crisis Briefing
*Report sitrep_758650c1b9 | generated 2026-11-21T19:11:25Z | data run run_20261111_201201Z*

> **DRAFT: requires human review before any public use.** AI-generated from verified sources. A person must review this before public release.

## Bottom line
15 active event(s): 3 priority, 0 confirmed, 15 preliminary, 0 with conflicting sources.

## Priority events
- weather alert | Lassen Volcanic National Park, Sierra Nevada of Sierra-Nevada-Placer Counties Above 5000 ft Including Interstate 80, Sierra Nevada of El Dorado-Amador-Northern Alpine Counties Above 5000 ft Including Highway 50 and 1 more: high-severity NOAA alert (Winter Storm Watch)
- weather alert | Surprise Valley California, Northern Washoe County: high-severity NOAA alert (High Wind Watch)
- declared emergency | Calhoun (County), FL, Holmes (County), FL, Okaloosa (County), FL and 5 more: federal disaster declaration

## Changes since last briefing
- 14 new events, including: weather alert | Montgomery; weather alert | South-Central Shasta County Including Redding Metro, Northern Sacramento Valley in Tehama County, Central Glenn County Including Interstate 5 and 2 more; weather alert | Surprise Valley California, Northern Washoe County; weather alert | Burke, Renville, Northwest Ward and 1 more; weather alert | Grand Traverse Bay south of a line Grand Traverse Light to Norwood MI, Norwood MI to 5NM West of Mackinac Bridge including Little Traverse Bay, Sleeping Bear Point to Grand Traverse Light MI and 2 more
- 16 ended events, including: weather alert | Montgomery; weather alert | Clatsop County Coast, Tillamook County Coast, South Washington Coast; weather alert | Mercer, Putnam, Schuyler and 13 more; weather alert | Atchison, Miami, Linn and 23 more; weather alert | Sodus Bay to Mexico Bay NY, Mexico Bay NY to the St. Lawrence River
- declared emergency | Calhoun (County), FL, Holmes (County), FL, Okaloosa (County), FL and 5 more: now a priority (federal disaster declaration)
- earthquake: 8 -> 6 active (decreasing)

## Confirmed information
- None.

## Preliminary information (attribute to source; may change)
- **weather alert | Lassen Volcanic National Park, Sierra Nevada of Sierra-Nevada-Placer Counties Above 5000 ft Including Interstate 80, Sierra Nevada of El Dorado-Amador-Northern Alpine Counties Above 5000 ft Including Highway 50 and 1 more** _(preliminary)_
  - NOAA alert 'Winter Storm Watch' for Lassen Volcanic National Park; Sierra Nevada of Sierra-Nevada-Placer Counties Above 5000 ft Including Interstate 80; Sierra Nevada of El Dorado-Amador-Northern Alpine Counties Above 5000 ft Including Highway 50; Sierra Nevada of Calaveras-Tuolumne-Southern Alpine Counties Above 5000 ft.
  - Figures: casualties: not reported; affected population: not reported
  - Sources: NOAA National Weather Service Alerts (latest 2026-11-11T20:03:00Z)
- **weather alert | Surprise Valley California, Northern Washoe County** _(preliminary)_
  - NOAA alert 'High Wind Watch' for Surprise Valley California; Northern Washoe County.
  - Figures: casualties: not reported; affected population: not reported
  - Sources: NOAA National Weather Service Alerts (latest 2026-11-11T19:52:00Z)
- **declared emergency | Calhoun (County), FL, Holmes (County), FL, Okaloosa (County), FL and 5 more** _(preliminary)_
  - FEMA declaration 4955 (HURRICANE ISAIAS) for Calhoun (County), FL. FEMA declaration 4955 (HURRICANE ISAIAS) for Holmes (County), FL. FEMA declaration 4955 (HURRICANE ISAIAS) for Okaloosa (County), FL. FEMA declaration 4955 (HURRICANE ISAIAS) for Gulf (County), FL. FEMA declaration 4955 (HURRICANE ISAIAS) for Liberty (County), FL. (and 3 more related reports)
  - Figures: casualties: not reported; affected population: not reported
  - Sources: FEMA Open Data (latest 2026-10-27T00:00:00Z)
- **earthquake | 3 km ESE of Berkeley, CA** _(preliminary)_
  - USGS reports earthquake magnitude 1.42 near 3 km ESE of Berkeley, CA.
  - Figures: casualties: not reported; affected population: not reported
  - Sources: USGS Earthquake Hazards Program (latest 2026-11-11T19:48:06Z)
- **earthquake | 4 km SSW of Guánica, Puerto Rico** _(preliminary)_
  - USGS reports earthquake magnitude 2.41 near 4 km SSW of Guánica, Puerto Rico.
  - Figures: casualties: not reported; affected population: not reported
  - Sources: USGS Earthquake Hazards Program (latest 2026-11-11T19:43:52Z)
- **earthquake | 62 km ENE of Pedro Bay, Alaska** _(preliminary)_
  - USGS reports earthquake magnitude 0.7 near 62 km ENE of Pedro Bay, Alaska.
  - Figures: casualties: not reported; affected population: not reported
  - Sources: USGS Earthquake Hazards Program (latest 2026-11-11T19:37:49Z)
- **earthquake | 48 km NW of Ancud, Chile** _(preliminary)_
  - USGS reports earthquake magnitude 5 near 48 km NW of Ancud, Chile.
  - Figures: casualties: not reported; affected population: not reported
  - Sources: USGS Earthquake Hazards Program (latest 2026-11-11T19:21:17Z)
- **earthquake | 2 km S of Pastos, Puerto Rico** _(preliminary)_
  - USGS reports earthquake magnitude 1.7 near 2 km S of Pastos, Puerto Rico.
  - Figures: casualties: not reported; affected population: not reported
  - Sources: USGS Earthquake Hazards Program (latest 2026-11-11T19:18:47Z)
- **earthquake | 58 km NW of Petersville, Alaska** _(preliminary)_
  - USGS reports earthquake magnitude 1.8 near 58 km NW of Petersville, Alaska.
  - Figures: casualties: not reported; affected population: not reported
  - Sources: USGS Earthquake Hazards Program (latest 2026-11-11T19:17:28Z)
- **weather alert | Grand Traverse Bay south of a line Grand Traverse Light to Norwood MI, Norwood MI to 5NM West of Mackinac Bridge including Little Traverse Bay, Sleeping Bear Point to Grand Traverse Light MI and 2 more** _(preliminary)_
  - NOAA alert 'Small Craft Advisory' for Grand Traverse Bay south of a line Grand Traverse Light to Norwood MI; Norwood MI to 5NM West of Mackinac Bridge including Little Traverse Bay; Sleeping Bear Point to Grand Traverse Light MI; Point Betsie to Sleeping Bear Point MI; Manistee to Point Betsie MI.
  - Figures: casualties: not reported; affected population: not reported
  - Sources: NOAA National Weather Service Alerts (latest 2026-11-11T20:07:00Z)
- **weather alert | 5NM East of Mackinac Bridge to Presque Isle Light MI including Bois Blanc Island, Presque Isle Light to Sturgeon Pt MI Including Thunder Bay National Marine Sanctuary, Sturgeon Pt to Alabaster MI** _(preliminary)_
  - NOAA alert 'Small Craft Advisory' for 5NM East of Mackinac Bridge to Presque Isle Light MI including Bois Blanc Island; Presque Isle Light to Sturgeon Pt MI Including Thunder Bay National Marine Sanctuary; Sturgeon Pt to Alabaster MI.
  - Figures: casualties: not reported; affected population: not reported
  - Sources: NOAA National Weather Service Alerts (latest 2026-11-11T20:07:00Z)
- **weather alert | St Ignace to False Detour Channel** _(preliminary)_
  - NOAA alert 'Small Craft Advisory' for St Ignace to False Detour Channel.
  - Figures: casualties: not reported; affected population: not reported
  - Sources: NOAA National Weather Service Alerts (latest 2026-11-11T20:07:00Z)
- **weather alert | Montgomery** _(preliminary)_
  - NOAA alert 'Test Message' for Montgomery.
  - Figures: casualties: not reported; affected population: not reported
  - Sources: NOAA National Weather Service Alerts (latest 2026-11-11T20:04:55Z)
- **weather alert | South-Central Shasta County Including Redding Metro, Northern Sacramento Valley in Tehama County, Central Glenn County Including Interstate 5 and 2 more** _(preliminary)_
  - NOAA alert 'Wind Advisory' for South-Central Shasta County Including Redding Metro; Northern Sacramento Valley in Tehama County; Central Glenn County Including Interstate 5; Eastern Glenn County; Butte County Below 1000 ft.
  - Figures: casualties: not reported; affected population: not reported
  - Sources: NOAA National Weather Service Alerts (latest 2026-11-11T19:54:00Z)
- **weather alert | Burke, Renville, Northwest Ward and 1 more** _(preliminary)_
  - NOAA alert 'Special Weather Statement' for Burke; Renville; Northwest Ward; Central and Southern Ward/Minot Area.
  - Figures: casualties: not reported; affected population: not reported
  - Sources: NOAA National Weather Service Alerts (latest 2026-11-11T19:54:00Z)

## Conflicting reports (do not state as fact)
- None.

## Information not available
- weather alert | Lassen Volcanic National Park, Sierra Nevada of Sierra-Nevada-Placer Counties Above 5000 ft Including Interstate 80, Sierra Nevada of El Dorado-Amador-Northern Alpine Counties Above 5000 ft Including Highway 50 and 1 more: casualties, affected population not reported by any source
- weather alert | Surprise Valley California, Northern Washoe County: casualties, affected population not reported by any source
- declared emergency | Calhoun (County), FL, Holmes (County), FL, Okaloosa (County), FL and 5 more: casualties, affected population not reported by any source
- earthquake | 3 km ESE of Berkeley, CA: casualties, affected population not reported by any source
- earthquake | 4 km SSW of Guánica, Puerto Rico: casualties, affected population not reported by any source
- earthquake | 62 km ENE of Pedro Bay, Alaska: casualties, affected population not reported by any source
- earthquake | 48 km NW of Ancud, Chile: casualties, affected population not reported by any source
- earthquake | 2 km S of Pastos, Puerto Rico: casualties, affected population not reported by any source
- earthquake | 58 km NW of Petersville, Alaska: casualties, affected population not reported by any source
- weather alert | Grand Traverse Bay south of a line Grand Traverse Light to Norwood MI, Norwood MI to 5NM West of Mackinac Bridge including Little Traverse Bay, Sleeping Bear Point to Grand Traverse Light MI and 2 more: casualties, affected population not reported by any source
- weather alert | 5NM East of Mackinac Bridge to Presque Isle Light MI including Bois Blanc Island, Presque Isle Light to Sturgeon Pt MI Including Thunder Bay National Marine Sanctuary, Sturgeon Pt to Alabaster MI: casualties, affected population not reported by any source
- weather alert | St Ignace to False Detour Channel: casualties, affected population not reported by any source
- weather alert | Montgomery: casualties, affected population not reported by any source
- weather alert | South-Central Shasta County Including Redding Metro, Northern Sacramento Valley in Tehama County, Central Glenn County Including Interstate 5 and 2 more: casualties, affected population not reported by any source
- weather alert | Burke, Renville, Northwest Ward and 1 more: casualties, affected population not reported by any source

## For the reviewer
- weather alert | Lassen Volcanic National Park, Sierra Nevada of Sierra-Nevada-Placer Counties Above 5000 ft Including Interstate 80, Sierra Nevada of El Dorado-Amador-Northern Alpine Counties Above 5000 ft Including Highway 50 and 1 more: model output failed grounding checks; showing source text
- weather alert | Lassen Volcanic National Park, Sierra Nevada of Sierra-Nevada-Placer Counties Above 5000 ft Including Interstate 80, Sierra Nevada of El Dorado-Amador-Northern Alpine Counties Above 5000 ft Including Highway 50 and 1 more: contains preliminary information
- weather alert | Surprise Valley California, Northern Washoe County: model output failed grounding checks; showing source text
- weather alert | Surprise Valley California, Northern Washoe County: contains preliminary information
- declared emergency | Calhoun (County), FL, Holmes (County), FL, Okaloosa (County), FL and 5 more: model output failed grounding checks; showing source text
- declared emergency | Calhoun (County), FL, Holmes (County), FL, Okaloosa (County), FL and 5 more: contains preliminary information
- earthquake | 3 km ESE of Berkeley, CA: model output failed grounding checks; showing source text
- earthquake | 3 km ESE of Berkeley, CA: contains preliminary information
- earthquake | 4 km SSW of Guánica, Puerto Rico: model output failed grounding checks; showing source text
- earthquake | 4 km SSW of Guánica, Puerto Rico: contains preliminary information
- earthquake | 62 km ENE of Pedro Bay, Alaska: model output failed grounding checks; showing source text
- earthquake | 62 km ENE of Pedro Bay, Alaska: contains preliminary information
- earthquake | 48 km NW of Ancud, Chile: model output failed grounding checks; showing source text
- earthquake | 48 km NW of Ancud, Chile: contains preliminary information
- earthquake | 2 km S of Pastos, Puerto Rico: model output failed grounding checks; showing source text
- earthquake | 2 km S of Pastos, Puerto Rico: contains preliminary information
- earthquake | 58 km NW of Petersville, Alaska: model output failed grounding checks; showing source text
- earthquake | 58 km NW of Petersville, Alaska: contains preliminary information
- weather alert | Grand Traverse Bay south of a line Grand Traverse Light to Norwood MI, Norwood MI to 5NM West of Mackinac Bridge including Little Traverse Bay, Sleeping Bear Point to Grand Traverse Light MI and 2 more: model output failed grounding checks; showing source text
- weather alert | Grand Traverse Bay south of a line Grand Traverse Light to Norwood MI, Norwood MI to 5NM West of Mackinac Bridge including Little Traverse Bay, Sleeping Bear Point to Grand Traverse Light MI and 2 more: contains preliminary information
- weather alert | 5NM East of Mackinac Bridge to Presque Isle Light MI including Bois Blanc Island, Presque Isle Light to Sturgeon Pt MI Including Thunder Bay National Marine Sanctuary, Sturgeon Pt to Alabaster MI: model output failed grounding checks; showing source text
- weather alert | 5NM East of Mackinac Bridge to Presque Isle Light MI including Bois Blanc Island, Presque Isle Light to Sturgeon Pt MI Including Thunder Bay National Marine Sanctuary, Sturgeon Pt to Alabaster MI: contains preliminary information
- weather alert | St Ignace to False Detour Channel: model output failed grounding checks; showing source text
- weather alert | St Ignace to False Detour Channel: contains preliminary information
- weather alert | Montgomery: model output failed grounding checks; showing source text
- weather alert | Montgomery: contains preliminary information
- weather alert | South-Central Shasta County Including Redding Metro, Northern Sacramento Valley in Tehama County, Central Glenn County Including Interstate 5 and 2 more: model output failed grounding checks; showing source text
- weather alert | South-Central Shasta County Including Redding Metro, Northern Sacramento Valley in Tehama County, Central Glenn County Including Interstate 5 and 2 more: contains preliminary information
- weather alert | Burke, Renville, Northwest Ward and 1 more: model output failed grounding checks; showing source text
- weather alert | Burke, Renville, Northwest Ward and 1 more: contains preliminary information


## 4. Public statement draft and safety check

In [5]:
draft = draft_public_statement(report, max_events=4)
print(draft["text"], "\n")
print("state:", draft["state"])
print(json.dumps(draft["safety"], indent=2))

Preliminary information from NOAA National Weather Service Alerts indicates a weather alert affecting Lassen Volcanic National Park, Sierra Nevada of Sierra-Nevada-Placer Counties Above 5000 ft Including Interstate 80, Sierra Nevada of El Dorado-Amador-Northern Alpine Counties Above 5000 ft Including Highway 50 and 1 more. No casualty figures have been confirmed.

Preliminary information from NOAA National Weather Service Alerts indicates a weather alert affecting Surprise Valley California, Northern Washoe County. No casualty figures have been confirmed.

Preliminary information from FEMA Open Data indicates a declared emergency affecting Calhoun (County), FL, Holmes (County), FL, Okaloosa (County), FL and 5 more. No casualty figures have been confirmed.

Preliminary information from USGS Earthquake Hazards Program indicates an earthquake near 3 km ESE of Berkeley, CA. No casualty figures have been confirmed.

Officials are also monitoring 11 other lower-priority alerts and reports.



### Safety check examples
Unsupported casualty numbers, vague estimates and unsourced numbers are blocked.

In [6]:
for text in ["Flooding has killed 12 people.", "Dozens of residents are injured.",
             "Shelters are open at 9 locations.", "Officials ask everyone to follow local guidance."]:
    res = check_release_safety(text, report)
    print("PASS   " if res["passed"] else "BLOCKED", text, res["blocking"][:1])

BLOCKED Flooding has killed 12 people. ["unverified casualty/population figure '12' in: 'Flooding has killed 12 people.'"]
BLOCKED Dozens of residents are injured. ["vague estimate 'Dozens of residents' is not a verified figure"]
BLOCKED Shelters are open at 9 locations. ["number '9' is not in the verified sources: 'Shelters are open at 9 locations.'"]
PASS    Officials ask everyone to follow local guidance. []


## 5. Human approval before release

In [7]:
try:
    release(draft, "comms-team")
except ReleaseBlocked as e:
    print("Release before approval blocked:", e)

approve(draft, "Duty Officer", "checked against sources")
edit_draft(draft, draft["text"] + " Stay tuned to local officials.", "Comms Editor")
print("state after edit:", draft["state"])          # edit cancels the approval

try:
    edit_draft(draft, draft["text"] + " At least 40 people are missing.", "Comms Editor")
    approve(draft, "Duty Officer")
except ReleaseBlocked as e:
    print("Unsafe edit cannot be approved:", e)

edit_draft(draft, draft_public_statement(report, max_events=4)["text"], "Comms Editor")  # back to the safe text
approve(draft, "Duty Officer", "final check")
released = release(draft, "comms-team")
for h in released["history"]:
    print(h["at_utc"], h["action"], "by", h["by"], "-", h["note"])

Release before approval blocked: draft is pending_review, not approved
state after edit: pending_review
Unsafe edit cannot be approved: fix blocking safety problems first: unverified casualty/population figure '40' in: 'At least 40 people are missing.'
2026-11-21T19:11:25Z created by system - 
2026-11-21T19:11:25Z approved by Duty Officer - checked against sources
2026-11-21T19:11:25Z edited by Comms Editor - approval reset
2026-11-21T19:11:25Z edited by Comms Editor - approval reset
2026-11-21T19:11:25Z edited by Comms Editor - approval reset
2026-11-21T19:11:25Z approved by Duty Officer - final check
2026-11-21T19:11:25Z released by comms-team - 
